In [31]:
#%pip install pandas numpy matplotlib

**LIBRARIES**

In [6]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

**PLOTS**

In [7]:
class DEPlot: 
    def __init__(self):
        self.name = "Puntos explorados"
        self.limits = [(0, 55), (0, 55)] # Ajustamos de 0 a 55 para cubrir todo el rango y dejar margen
    
    def lectura(self, filepath: str):
        # 1. Cargar datos 
        # Leemos el archivo detectando automáticamente el separador (puede ser coma, punto y coma, tabulación, etc.)
        df = pd.read_csv(filepath, sep=None, engine='python')

        # Limpiamos nombres de columnas (quita espacios invisibles como " X" -> "X")
        df.columns = df.columns.str.strip()
        print(f"Columnas detectadas: {list(df.columns)}")

        # 3. Extraer valores de la columna 2 (X) y de la columna 3(Y) (Usando .iloc es más seguro si las columnas son la 2 y la 3)
        val_x = df.iloc[:, 1] # iloc[:, 1] es la segunda columna donde estan los valores de Soluplus 
        val_y = df.iloc[:, 2] # iloc[:, 2] es la tercera columna donde estan los valores de TPGS 

        return val_x, val_y, df

In [8]:
def plot_DE(val_x, val_y, round: str): #, save_figure: bool = False): 

    
    ###-PLOT-###
    # 1. Crear el fondo de la función
    x_range = np.linspace(0, 55, 400)
    y_range = np.linspace(0, 55, 400)
    X_mesh, Y_mesh = np.meshgrid(x_range, y_range)

    # 2. Dibujar el mapa
    plt.figure(figsize=(10, 7))
    plt.contourf(X_mesh, Y_mesh, Z_mesh, levels=np.linspace(np.nanmin(val_z), np.nanmax(val_z), 35), norm=LogNorm(), cmap='viridis', alpha=0.7)
    plt.colorbar(label='Himmelblau value')

    # 3. Dibujar TUS puntos (usando el df que ya tienes en memoria)
    plt.scatter(val_x, val_y, color='red', edgecolors='white', s=80, label='Explored points')

    # Añadir etiquetas de número de RUN
    #for i, row in df.iterrows():
    #    plt.text(row.iloc[1]+0.1, row.iloc[2]+0.1, str(int(row.iloc[0])), color='white', fontsize=9)

    plt.title(f"Round {round}")
    plt.xlabel("Soluplus (mg/mL)")
    plt.ylabel("TPGS (mg/mL)")
    plt.legend()

    return plt

In [9]:
csvs = [f for f in os.listdir(os.getcwd()) if f.endswith('.csv') and f.startswith('Design_Expert') and "sampling" not in f and "plot" not in f]

for file in csvs:
    output_file = file.replace('.csv', '_RESULTADO.csv')
    plot_file   = file.replace('.csv', '_plot.jpg')
    round       = int(file.split('_')[-1].replace('.csv', '').replace('R', '')) # Extrae el número de round del nombre del archivo

    print("-" * 30)
    print(f"Procesando archivo: {file}")
    print(f"Guardando Output en {output_file}")
    print(f"Guardando Gráfico en {plot_file}")
    print(f"Round: {round}")
    print("-" * 30)

    val_x, val_y, df = lectura(file)

    # 4. Calcular el resultado y gua1rdarlo en la nueva columna 4 (indice 3)
    df['R0'] = function.evaluate(val_x, val_y)

    # 5. Guardar el archivo actualizado en el Escritorio
    df.to_csv(output_file, index=False)

    print(df.head()) # Mostrar los primeros resultados para verificar
    plt = plot_himmelblau(val_x, val_y, str(round))
    
    if save_plot:
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"Gráfico guardado correctamente como :{plot_file}")



In [1]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
from scipy.interpolate import griddata # Necesario para el fondo

# --- CONFIGURACIÓN ---
file_name = "Design_Expert-sampling.csv" # Tu archivo específico
save_plot = True

def dibujar_puntos_reales(filepath):
    # 1. Leer datos
    # Usamos sep=None para que detecte si es coma o punto y coma automáticamente
    df = pd.read_csv(filepath, sep=None, engine='python')
    df.columns = df.columns.str.strip()
    
    # 2. Extraer columnas (2=Soluplus, 3=TPGS, 4=Respuesta)
    val_x = df.iloc[:, 1] # Columna 2
    val_y = df.iloc[:, 2] # Columna 3
    val_z = df.iloc[:, 3] # Columna 4 (el resultado que da el color)

    # 3. Crear malla para el fondo (0 a 55 mg/mL)
    xi = np.linspace(0, 55, 500)
    yi = np.linspace(0, 55, 500)
    X_mesh, Y_mesh = np.meshgrid(xi, yi)

    # 4. Interpolación (Crea el "mapa de calor" basado en tus puntos)
    # Si tienes pocos puntos, usa method='linear'. Para curvas suaves: 'cubic'
    Z_mesh = griddata((val_x, val_y), val_z, (X_mesh, Y_mesh), method='linear')

    # 5. Crear el Gráfico
    plt.figure(figsize=(10, 7))
    
    # Fondo de color
    # Nota: He quitado LogNorm porque para datos reales suele ser mejor escala lineal
    cp = plt.contourf(X_mesh, Y_mesh, Z_mesh, levels=20, cmap='viridis', alpha=0.8)
    plt.colorbar(cp, label='Respuesta Experimental')

    # Puntos rojos (tus muestras)
    plt.scatter(val_x, val_y, color='red', edgecolors='white', s=100, label='Puntos Design-Expert')

    # Etiquetas
    plt.title("Visualización de Formulación: Soluplus vs TPGS")
    plt.xlabel("Soluplus (mg/mL)")
    plt.ylabel("TPGS (mg/mL)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    # Guardar y mostrar
    if save_plot:
        plt.savefig("grafico_design_expert.jpg", dpi=300, bbox_inches='tight')
        print("Gráfico guardado como grafico_design_expert.jpg")
    
    plt.show()

# Ejecutar
if os.path.exists(file_name):
    dibujar_puntos_reales(file_name)
else:
    print(f"No se encontró el archivo {file_name} en la carpeta actual.")

No se encontró el archivo Design_Expert-sampling.csv en la carpeta actual.


In [5]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
from scipy.interpolate import griddata


filepath = "emma-reig/TFM_MuBiB_Emma-Reig/Design_Expert-sampling.csv"

# --- CONFIGURACIÓN ---
file_name = "Design_Expert-sampling.csv" 
save_plot = True

def dibujar_puntos_reales(filepath):
    # 1. Leer datos
    df = pd.read_csv(filepath, sep=None, engine='python')
    df.columns = df.columns.str.strip()
    
    # 2. Extraer columnas por posición
    val_x = df.iloc[:, 1] # Columna 2: Soluplus
    val_y = df.iloc[:, 2] # Columna 3: TPGS
    
    # Intentamos obtener la columna 4, si no existe o es nula, solo dibujamos puntos
    try:
        val_z = df.iloc[:, 3]
        tiene_resultados = not val_z.isnull().all() and not (val_z == 0).all()
    except IndexError:
        tiene_resultados = False

    # 3. Preparar el gráfico
    plt.figure(figsize=(10, 7))
    
    if tiene_resultados:
        # Creamos el fondo interpolado
        xi = np.linspace(0, 55, 500)
        yi = np.linspace(0, 55, 500)
        X_mesh, Y_mesh = np.meshgrid(xi, yi)
        
        # 'linear' es más seguro para pocos puntos; 'cubic' es más estético
        Z_mesh = griddata((val_x, val_y), val_z, (X_mesh, Y_mesh), method='linear')
        
        cp = plt.contourf(X_mesh, Y_mesh, Z_mesh, levels=20, cmap='viridis', alpha=0.8)
        plt.colorbar(cp, label='Respuesta (Columna 4)')
        plt.title("Mapa de Calor: Resultados Experimentales")
    else:
        # Si no hay datos en la Columna 4, solo configuramos el área de dibujo
        plt.xlim(0, 55)
        plt.ylim(0, 55)
        plt.title("Plan de Muestreo: Puntos a Explorar")

    # 4. Dibujar los puntos (siempre se dibujan)
    plt.scatter(val_x, val_y, color='red', edgecolors='white', s=100, zorder=5, label='Muestras')

    # 5. Estética final
    plt.xlabel("Soluplus (mg/mL)")
    plt.ylabel("TPGS (mg/mL)")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.legend()

    if save_plot:
        nombre_salida = "grafico_muestreo.jpg"
        plt.savefig(nombre_salida, dpi=300, bbox_inches='tight')
        print(f"Gráfico guardado como {nombre_salida}")
    
    plt.show()

if os.path.exists(file_name):
    dibujar_puntos_reales(file_name)
else:
    print(f"Archivo '{file_name}' no encontrado. Verifica que esté en la misma carpeta que este código.")

Archivo 'Design_Expert-sampling.csv' no encontrado. Verifica que esté en la misma carpeta que este código.
